In [1]:
!pip install minsearch


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [3]:
from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

### Implementing Search


In [4]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

### Prompt

In [5]:
question = 'can I still join the course?'

In [6]:
prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [7]:
search_results = search(question)

In [8]:
prompt = build_prompt(question, search_results)

In [9]:
print(prompt)

You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

<QUESTION>
can I still join the course?
</QUESTION>

<CONTEXT>
section: General course-related questions
question: Course - Can I still join the course after the start date?
answer: Yes, even if you don't register, you're still eligible to submit the homeworks.
Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.

section: General course-related questions
question: Certificate - Can I follow the course in a self-paced mode and get a certificate?
answer: No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.

section: General

### The RAG flow

In [10]:
from openai import OpenAI
client = OpenAI(api_key="your_api_key", base_url="https://api.groq.com/openai/v1")

def llm(prompt):
    response = client.chat.completions.create(
    model='llama3-70b-8192',
    messages=[{"role":"user", "content": prompt}]
    )
    return response.choices[0].message.content



In [11]:
answer = llm(prompt)

In [12]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [13]:
rag("What is grad school?")

"I'm happy to help!\n\nSince the context is empty, I'll provide a general answer based on common knowledge.\n\nGrad school, also known as graduate school, is a type of educational institution or program that provides advanced academic degrees, such as master's or doctoral degrees, to students who have already completed their undergraduate studies. These programs are designed to provide specialized knowledge and training in a specific field or profession, and often involve original research, coursework, and other academic requirements."

In [14]:
print(llm("What is grad school?"))

Grad school, short for graduate school, is a type of educational institution that offers advanced degrees beyond a bachelor's degree. It's a place where students can pursue higher-level education and training in a specific field or discipline.

Grad school typically offers the following types of degrees:

1. **Master's degree**: A master's degree is a postgraduate degree that usually takes two to three years to complete. It provides advanced knowledge and skills in a specific field, and is often required for certain careers or professional roles.
2. **Doctoral degree** (Ph.D.): A doctoral degree is the highest academic degree in a field, typically taking four to six years to complete. It involves original research, academic coursework, and the production of a dissertation or thesis.

Grad school programs often have different formats, such as:

1. **On-campus programs**: Students attend classes and work with faculty on a university campus.
2. **Online programs**: Students complete cours

### What I illustrated above is a major limitation of RAG
- Can not answer questions out of the scope of the Knowledge base. And this is where Agentic RAG comes in.

## "Agentic" RAG

In [15]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
{question}
</QUESTION>

<CONTEXT> 
{context}
</CONTEXT>

If CONTEXT is EMPTY, you can use our FAQ database.
In this case, use the following output template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>"
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}
""".strip()

In [16]:
question = "can I still join the course?"
context = "EMPTY"

In [17]:
prompt = prompt_template.format(question=question, context=context)


In [18]:
print(prompt)

You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
can I still join the course?
</QUESTION>

<CONTEXT> 
EMPTY
</CONTEXT>

If CONTEXT is EMPTY, you can use our FAQ database.
In this case, use the following output template:

{
"action": "SEARCH",
"reasoning": "<add your reasoning here>"
}

If you can answer the QUESTION using CONTEXT, use this template:

{
"action": "ANSWER",
"answer": "<your answer>",
"source": "CONTEXT"
}

If the context doesn't contain the answer, use your own knowledge to answer the question

{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}


In [19]:
print(llm(prompt))

{
"action": "SEARCH",
"reasoning": "The context is empty, so I'll search the FAQ database to see if I can find an answer to the student's question about joining the course."
}


In [20]:
answer_json = llm(prompt)

In [21]:
import json

In [22]:
answer = json.loads(answer_json)

In [23]:
answer['action']

'SEARCH'

In [24]:
def build_context(search_results):
    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"

    return context.strip()

In [25]:
def agentic_rag_v1(question):
    context = "EMPTY"
    prompt = prompt_template.format(question=question, context=context)
    answer_json = llm(prompt)
    answer = json.loads(answer_json)
    print(answer)

    if answer['action'] == 'SEARCH':
        print('need to perform search...')
        search_results = search(question)
        context = build_context(search_results)
        
        prompt = prompt_template.format(question=question, context=context)
        answer_json = llm(prompt)
        answer = json.loads(answer_json)
        print(answer)

    return answer

In [26]:
agentic_rag_v1('how do I join the course?')


{'action': 'SEARCH', 'reasoning': "The student is asking how to join the course, which is a common question that is typically covered in the course FAQ. I'll search the FAQ database for the answer."}
need to perform search...
{'action': 'ANSWER', 'answer': "To join the course, register before the course starts using this link. Also, subscribe to the course public Google Calendar (it works from Desktop only), join the course Telegram channel with announcements, and don't forget to register in DataTalks.Club's Slack and join the channel.", 'source': 'CONTEXT'}


{'action': 'ANSWER',
 'answer': "To join the course, register before the course starts using this link. Also, subscribe to the course public Google Calendar (it works from Desktop only), join the course Telegram channel with announcements, and don't forget to register in DataTalks.Club's Slack and join the channel.",
 'source': 'CONTEXT'}

In [27]:
agentic_rag_v1('how patch KDE under FreeBSD?')

{'action': 'SEARCH', 'reasoning': 'The context is empty, so I need to search in the FAQ database to find the answer on how to patch KDE under FreeBSD.'}
need to perform search...
{'action': 'SEARCH', 'reasoning': "I don't have enough information in the context to answer the question, so I'll search the FAQ database to see if there's a relevant answer on how to patch KDE under FreeBSD."}


{'action': 'SEARCH',
 'reasoning': "I don't have enough information in the context to answer the question, so I'll search the FAQ database to see if there's a relevant answer on how to patch KDE under FreeBSD."}

## AgentiC Search

In [28]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than {max_iterations} iterations for a given student question.
The current iteration number: {iteration_number}. If we exceed the allowed number 
of iterations, give the best possible answer with the provided information.

Output templates:

If you want to perform search, use this template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>",
"keywords": ["search query 1", "search query 2", ...]
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER_CONTEXT",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}

<QUESTION>
{question}
</QUESTION>

<SEARCH_QUERIES>
{search_queries}
</SEARCH_QUERIES>

<CONTEXT> 
{context}
</CONTEXT>

<PREVIOUS_ACTIONS>
{previous_actions}
</PREVIOUS_ACTIONS>
""".strip()

In [29]:
question = "how do I join the course?"

search_queries = []
search_results = []
previous_actions = []
context = build_context(search_results)

prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=3,
    iteration_number=1
)
print(prompt)

You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than 3 iterations for a given student question.
The current iteration number

In [30]:
answer_json = llm(prompt)
answer = json.loads(answer_json)
print(json.dumps(answer, indent=2))

{
  "action": "SEARCH",
  "reasoning": "The student is asking how to join the course, so I need to search for relevant information about course enrollment, registration, or participation.",
  "keywords": [
    "course enrollment",
    "join the course",
    "register for the course"
  ]
}


In [31]:
previous_actions.append(answer)

In [32]:
keywords = answer['keywords']
search_queries.extend(keywords)

In [33]:
for k in keywords:
    res = search(k)
    search_results.extend(res)

In [34]:
def dedup(seq):
    seen = set()
    result = []
    for el in seq:
        _id = el['_id']
        if _id in seen:
            continue
        seen.add(_id)
        result.append(el)
    return result

search_results = dedup(search_results)

In [35]:
# question = "how do I join the course?"

# search_queries = []
# search_results = []
# previous_actions = []

context = build_context(search_results)

prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=3,
    iteration_number=2
)
print(prompt)

answer_json = llm(prompt)
answer = json.loads(answer_json)
print(json.dumps(answer, indent=2))

You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than 3 iterations for a given student question.
The current iteration number

JSONDecodeError: Extra data: line 7 column 1 (char 137)

In [36]:
def agentic_search(question):
    search_queries = []
    search_results = []
    previous_actions = []

    iteration = 0
    
    while True:
        print(f'ITERATION #{iteration}...')
    
        context = build_context(search_results)
        prompt = prompt_template.format(
            question=question,
            context=context,
            search_queries="\n".join(search_queries),
            previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
            max_iterations=3,
            iteration_number=iteration
        )
    
        print(prompt)
    
        answer_json = llm(prompt)
        answer = json.loads(answer_json)
        print(json.dumps(answer, indent=2))

        previous_actions.append(answer)
    
        action = answer['action']
        if action != 'SEARCH':
            break
    
        keywords = answer['keywords']
        search_queries = list(set(search_queries) | set(keywords))

        for k in keywords:
            res = search(k)
            search_results.extend(res)
    
        search_results = dedup(search_results)
        
        iteration = iteration + 1
        if iteration >= 4:
            break
    
        print()

    return answer

In [37]:
agentic_search('how do I prepare for the course?')

ITERATION #0...
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than 3 iterations for a given student question.
The current 

{'action': 'ANSWER_CONTEXT',
 'answer': 'To prepare for the course, you can start by installing and setting up all the dependencies and requirements: Google cloud account, Google Cloud SDK, Python 3 (installed with Anaconda), Terraform, and Git. Also, look over the prerequisites and syllabus to see if you are comfortable with these subjects.',
 'source': 'CONTEXT'}

## Part 3: Function calling

In [38]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

In [39]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}


In [40]:
question = "How do I do well in module 1?"

developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.
""".strip()

tools = [search_tool]

chat_messages = [
    {"role": "developer", "content": developer_prompt},
    {"role": "user", "content": question}
]

response = client.chat.completions.create(
    model="llama3-70b-8192",  # or whichever Groq-supported model you choose
    messages=chat_messages,
    tools=tools
)

print(response.choices[0].message)

tool_call = response.choices[0].message.tool_calls[0]

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='0jc7hrebb', function=Function(arguments='{"query":"How do I do well in module 1"}', name='search'), type='function')])


In [41]:
# Make request

tool_calls = response.choices[0].message.tool_calls
tool_call = tool_calls[0]

# Get call id
call_id = tool_call.id
print("Call ID:", call_id)

# Get function name
f_name = tool_call.function.name
print("Function name:", f_name)

# Get arguments as dict
arguments = json.loads(tool_call.function.arguments)
print("Arguments:", arguments)


Call ID: 0jc7hrebb
Function name: search
Arguments: {'query': 'How do I do well in module 1'}


In [42]:
f = globals()[f_name]

In [43]:
results = f(**arguments)

In [44]:
search_results = json.dumps(results, indent=2)
print(search_results)

[
  {
    "text": "Following dbt with BigQuery on Docker readme.md, after `docker-compose build` and `docker-compose run dbt-bq-dtc init`, encountered error `ModuleNotFoundError: No module named 'pytz'`\nSolution:\nAdd `RUN python -m pip install --no-cache pytz` in the Dockerfile under `FROM --platform=$build_for python:3.9.9-slim-bullseye as base`",
    "section": "Module 4: analytics engineering with dbt",
    "question": "DBT - Error: No module named 'pytz' while setting up dbt with docker",
    "course": "data-engineering-zoomcamp",
    "_id": 299
  },
  {
    "text": "Even after installing pyspark correctly on linux machine (VM ) as per course instructions, faced a module not found error in jupyter notebook .\nThe solution which worked for me(use following in jupyter notebook) :\n!pip install findspark\nimport findspark\nfindspark.init()\nThereafter , import pyspark and create spark contex<<t as usual\nNone of the solutions above worked for me till I ran !pip3 install pyspark inst

In [45]:
chat_messages.append(tool_call)

chat_messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,  # equivalent to call_id
    "content": json.dumps(search_results)  # output must be stringified JSON
})
# Append the assistant's message containing the tool call
chat_messages.append({
    "role": "assistant",
    "tool_calls": [{
        "id": tool_call.id,  # ID of the tool call from the previous step
        "type": "function",
        "function": {
            "name": tool_call.function.name,
            "arguments": tool_call.function.arguments
        }
    }]
})

# Append the tool's output message
chat_messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,  # Matches the ID above
    "content": json.dumps(search_results)  # Must be JSON string
})


In [46]:
# 4. Build new chat history — DO NOT append raw tool_call object
chat_messages = [
    {"role": "system", "content": developer_prompt},
    {"role": "user", "content": question},
    {
        "role": "assistant",
        "tool_calls": [{
            "id": tool_call.id,
            "type": "function",
            "function": {
                "name": f_name,
                "arguments": tool_call.function.arguments
            }
        }]
    },
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(results)
    },
    {
        "role": "user",
        "content": "Please use the tool output above to fully answer my original question."
    }
]

# 5. Second call — get final answer
final_response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=chat_messages
)

print(final_response.choices[0].message.content)

I apologize for the confusion earlier!

To answer your original question, "How do I do well in Module 1?", I'll provide some general advice and tips based on common issues students faced in Module 1 of the data-engineering-zoomcamp course.

**General Tips:**

1. **Carefully follow the instructions**: Make sure you read and follow the instructions provided in the course materials, especially when setting up your environment with Docker and Terraform.
2. **Install required dependencies**: Pay attention to the installation steps for dependencies like `psycopg2` and `pytz`. If you encounter errors, check if you have installed these packages correctly.
3. **Use the correct versions**: Be mindful of the versions of dependencies and tools you are using. If you encounter errors, check if you are using the correct versions.

**Specific Tips based on Common Issues:**

1. **Postgres and psycopg2**: If you encounter `ModuleNotFoundError: No module named 'psycopg2'`, try installing `psycopg2-binary

### Making Multiple Calls

In [48]:
import json

developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.
If you look up something in FAQ, convert the student question into multiple queries.
""".strip()

chat_messages = [
    {"role": "system", "content": developer_prompt},
    {"role": "user", "content": question}
]

# Step 1: Ask the model
response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=chat_messages,
    tools=tools
)

# Extract tool call request
tool_call = response.choices[0].message.tool_calls[0]
f_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

# Step 2: Run the tool in Python
results = globals()[f_name](**arguments)

# Step 3: Append assistant tool call message
chat_messages.append({
    "role": "assistant",
    "tool_calls": [{
        "id": tool_call.id,
        "type": "function",
        "function": {
            "name": f_name,
            "arguments": json.dumps(arguments)
        }
    }]
})

# Step 4: Append tool output
chat_messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(results)
})

# Step 5: Ask model to continue with the tool output
final_response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=chat_messages
)

print(final_response.choices[0].message.content)


It seems like the tool yielded results that are not directly related to your question "How do I do well in module 1?" 

Instead, the results appear to be related to specific error troubleshooting in various modules, including Module 1: Docker and Terraform, Module 4: analytics engineering with dbt, and Module 5: pyspark.

To better answer your question, here are some general tips on how to do well in Module 1:

* Carefully follow the instructions and tutorials provided in the module.
* Make sure you have a good understanding of the basics of Docker and Terraform.
* Practice and experiment with the concepts learned in the module.
* Join online communities or discussion forums related to the course to ask questions and get help from instructors and peers.
* Review the module's materials and take notes to reinforce your learning.

If you have any specific questions or need help with a particular topic in Module 1, feel free to ask, and I'll do my best to assist you!


In [49]:
def do_call(tool_call_response):
    # Get function name and arguments
    function_name = tool_call_response.function.name
    arguments = json.loads(tool_call_response.function.arguments)

    # Dynamically call the function
    f = globals()[function_name]
    result = f(**arguments)

    # Return in Groq's format (no "type": "function_call_output")
    return {
        "role": "tool",  # Groq expects a 'role'
        "tool_call_id": tool_call_response.id,  # equivalent to call_id
        "content": json.dumps(result, indent=2)  # output as JSON string
    }


In [50]:
for choice in response.choices:
    msg = choice.message
    chat_messages.append(msg)

    # Check if the model made a tool call
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tool_call in msg.tool_calls:
            result = do_call(tool_call)  # uses the Groq do_call we just made
            chat_messages.append(result)
    else:
        # Just a normal assistant message
        print(msg.content)


In [51]:
# Second call to continue conversation
response = client.chat.completions.create(
    model="llama3-70b-8192",  # Or whichever Groq-supported model you're using
    messages=chat_messages,
    tools=tools
)

for choice in response.choices:
    msg = choice.message
    chat_messages.append(msg)

    # Debug print
    print("Message role:", msg.role)
    print()

    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tool_call in msg.tool_calls:
            result = do_call(tool_call)  # Calls our Groq version of do_call
            chat_messages.append(result)
    else:
        print(msg.content)


Message role: assistant

It seems like the results from the tool call were not relevant to your question "How do I do well in module 1?". The results were related to error solutions for different modules and topics.

To answer your original question, doing well in Module 1 of the data engineering zoomcamp course likely requires:

* Carefully following the course instructions and setup for Docker and Terraform
* Ensuring you have the correct installations and configurations for the course materials
* Practicing and completing the exercises and assignments provided in the module
* Asking for help and clarification when needed

However, without more specific information about the module and its contents, it's difficult to provide more detailed advice.


In [52]:
developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

Use FAQ if your own knowledge is not sufficient to answer the question.
When using FAQ, perform deep topic exploration: make one request to FAQ,
and then based on the results, make more requests.

At the end of each response, ask the user a follow up question based on your answer.
""".strip()

# Groq chat messages list
chat_messages = [
    {"role": "system", "content": developer_prompt}  # Groq uses 'system' instead of 'developer'
]


In [ ]:
while True:  # main Q&A loop
    question = input("Ask a question (type 'stop' to end): ")
    if question.lower() == 'stop':
        break

    # Append the user's message
    chat_messages.append({
        "role": "user",
        "content": question
    })

    while True:  # request-response loop - query API until we get a message
        response = client.chat.completions.create(
            model="llama3-70b-8192",  # Groq model
            messages=chat_messages,
            tools=tools
        )

        has_tool_calls = False
        message = response.choices[0].message
        chat_messages.append(message)

        # Handle tool calls
        if hasattr(message, "tool_calls") and message.tool_calls:
            for tool_call in message.tool_calls:
                print("function_call:", tool_call)
                print()

                result = do_call(tool_call)  # Your function to run the tool
                chat_messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result["output"]
                })
                has_tool_calls = True

        # Handle normal assistant message
        elif message.content:
            print(message.content)
            print()

        # Exit request loop if no more tool calls
        if not has_tool_calls:
            break


### Using Multiple tools

In [61]:
def add_entry(question, answer):
    doc = {
        "question": question,
        "text": answer,
        "section": "user added",
        "course": "data-engineering-zoomcamp"
    }
    index.append(doc)  # index is your existing database or list


In [62]:
add_entry_description = {
    "type": "function",
    "function": {  # Groq needs the 'function' key here
        "name": "add_entry",
        "description": "Add an entry to the FAQ database",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "The question to be added to the FAQ database"
                },
                "answer": {
                    "type": "string",
                    "description": "The answer to the question"
                }
            },
            "required": ["question", "answer"],
            "additionalProperties": False
        }
    }
}


In [63]:
# Just keep a list of tool descriptions
tools = [
    search_tool,          # assuming you defined this earlier
    add_entry_description # the one we converted above
]

# When calling the model:
response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=chat_messages,
    tools=tools
)


In [57]:
!pip install markdown


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [55]:
!wget https://raw.githubusercontent.com/alexeygrigorev/rag-agents-workshop/refs/heads/main/chat_assistant.py

--2025-08-11 09:48:21--  https://raw.githubusercontent.com/alexeygrigorev/rag-agents-workshop/refs/heads/main/chat_assistant.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3495 (3.4K) [text/plain]
Saving to: ‘chat_assistant.py’

chat_assistant.py   100%[===================>]   3.41K  --.-KB/s    in 0s      

2025-08-11 09:48:21 (35.8 MB/s) - ‘chat_assistant.py’ saved [3495/3495]



In [58]:
import chat_assistant

tools = chat_assistant.Tools()
tools.add_tool(search, search_tool)

tools.get_tools()

developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

Use FAQ if your own knowledge is not sufficient to answer the question.

At the end of each response, ask the user a follow up question based on your answer.
""".strip()

chat_interface = chat_assistant.ChatInterface()

chat = chat_assistant.ChatAssistant(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    client=client
)

In [ ]:
chat.run()

### Using PydanticAI

In [64]:
!pip install pydantic-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub╸━━━━━━━━━━━━━━━━━━━━━ 21/45 [starlette]
    Found existing installation: huggingface-hub 0.33.44m╸━━━━━━━━━━━━━━━━━━━━━ 21/45 [starlette]
    Uninstalling huggingface-hub-0.33.4:━━━╺━━━━━━━━━━━━━━━━━ 25/45 [huggingface-hub]
      Successfully uninstalled huggingface-hub-0.33.4m╺━━━━━━━━━━━━━━━━━ 25/45 [huggingface-hub]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45/45 [pydantic-ai]m 41/45 [cohere]mcp]ntic-ai-slim]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [65]:
    from pydantic_ai import Agent, RunContext

In [66]:
# Groq equivalent
chat_messages = [
    {"role": "system", "content": developer_prompt}
]

# Then you just call the model when needed
response = client.chat.completions.create(
    model="llama3-70b-8192",  # or another Groq-supported model
    messages=chat_messages
)


In [67]:
from typing import Dict
import json

# Define the actual Python functions
def search_tool(query: str) -> Dict[str, str]:
    """
    Search the FAQ for relevant entries matching the query.
    """
    print(f"search('{query}')")
    return search(query)  # your existing search() function

def add_entry_tool(question: str, answer: str) -> None:
    """
    Add a new question-answer entry to FAQ.
    """
    return add_entry(question, answer)

# Define tool descriptions manually for Groq
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tool",
            "description": "Search the FAQ for relevant entries matching the query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query string provided by the user."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_entry_tool",
            "description": "Add a new question-answer entry to FAQ.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "The question text to be added to the index."
                    },
                    "answer": {
                        "type": "string",
                        "description": "The answer or explanation corresponding to the question."
                    }
                },
                "required": ["question", "answer"]
            }
        }
    }
]

# Simulate "agent" behavior with a manual call
user_prompt = "I just discovered the course. Can I join now?"

chat_messages = [
    {"role": "system", "content": developer_prompt},
    {"role": "user", "content": user_prompt}
]

response = client.chat.completions.create(
    model="llama3-70b-8192",  # Groq-supported model
    messages=chat_messages,
    tools=tools
)

print(response.choices[0].message)


ChatCompletionMessage(content="Welcome to the course! Yes, you can definitely join now. Since we've already started, you might want to review the previous lectures and course materials to catch up. But don't worry, we're happy to have you on board.\n\n<tool-use></tool-use>\n\nCan you tell me a little bit about your background and what motivated you to join this course?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
